### 모델 학습 방법

1. 주피터 노트북에 접속합니다.
2. `00_MLOps_Training_Template.ipynb` 파일을 우클릭하여 **Duplicate(복제)** 합니다.
3. 복제된 파일의 이름을 `0X_CNN_Training.ipynb` 등으로 바꿉니다.
4. **[Cell 3]** 영역만 본인이 작성한 AI 모델 코드로 싹 수정합니다.
5. 위에서부터 아래로 셀을 실행(`Shift + Enter`)합니다.
6. 완료 후 플랫폼 웹 UI의 [History] 탭에 들어가면, 방금 주피터에서 돌린 학습 이력과 MLflow 지표가 다른 작업들과 동일하게 저장되어 있는 것을 확인합니다.

[Cell 1] 라이브러리 및 환경 설정

In [ ]:
import os
import requests
import mlflow
import json
import boto3
from ultralytics import YOLO, settings
import mlflow.pytorch

settings.update({"mlflow": False})

# 환경 설정
BACKEND_URL = "http://backend:8000"
MLFLOW_TRACKING_URI = "http://mlflow:5000"
S3_ENDPOINT_URL = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://minio:9000")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# 데이터셋 다운로드 함수
def download_dataset(bucket_name, prefix, local_dir="./data"):
    s3 = boto3.client('s3',
        endpoint_url=S3_ENDPOINT_URL,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY
    )
    if not os.path.exists(local_dir):
        os.makedirs(local_dir)
    print(f"Downloading: {bucket_name}/{prefix} -> {local_dir}")
    paginator = s3.get_paginator('list_objects_v2')
    for result in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if 'Contents' in result:
            for obj in result['Contents']:
                key = obj['Key']
                if key.endswith('/'): continue
                local_file_path = os.path.join(local_dir, os.path.relpath(key, prefix))
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
                s3.download_file(bucket_name, key, local_file_path)
    return local_dir

[Cell 2] 작업 등록 및 데이터 준비

In [ ]:
PROJECT_ID = 1  # UI에 등록된 프로젝트 ID
DATASET_PATH = "my-bucket/dataset-v1" 
MODEL_ARCHITECTURE = "custom_model_v1"

# 1. FastAPI 백엔드에 작업 등록
payload = {
    "project_id": PROJECT_ID,
    "model_variant": MODEL_ARCHITECTURE,
    "dataset": DATASET_PATH,
    "params": {"source": "jupyter_notebook"}
}
response = requests.post(f"{BACKEND_URL}/api/v1/jobs/jupyter", json=payload)
response.raise_for_status()
job_info = response.json()

JOB_ID = job_info["id"]
RUN_ID = job_info["run_id"]

print(f"[System] Job Registered! ID: {JOB_ID}, Run ID: {RUN_ID}")

[Cell 3] 모델 학습 (사용자가 자유롭게 수정하는 부분)

In [ ]:
# 사용자는 이 셀의 내용만 자신의 모델(YOLO, PyTorch, Tensorflow 등)에 맞게 수정합니다.

with mlflow.start_run(run_id=RUN_ID):
    print("[System] Training Started...")
    
    try:
        # 1. MinIO에서 데이터셋 다운로드 (다운로드 실패 시에도 FAILED 처리를 위해 try 블록 내부로 이동)
        bucket, prefix = DATASET_PATH.split('/', 1)
        local_data_path = download_dataset(bucket, prefix, local_dir=f"./data/job_{JOB_ID}")
        print(f"[System] Dataset downloaded at: {local_data_path}")

        # 2. --- 사용자 커스텀 모델 학습 코드 로직 ---
        # model.fit(...)
        # mlflow.log_metric("accuracy", 0.99)
        # mlflow.pytorch.log_model(model, "model")
        # ----------------------------------------
        
        print("[System] Training Complete!")
        status_to_report = "FINISHED"
        message = "Jupyter training completed successfully."

    except Exception as e:
        print(f"[Error] Training Failed: {e}")
        status_to_report = "FAILED"
        message = str(e)
        mlflow.end_run(status='FAILED')

[Cell 4] 결과 전송

In [ ]:
webhook_url = f"{BACKEND_URL}/api/v1/jobs/{JOB_ID}/complete"
resp = requests.post(webhook_url, json={"status": status_to_report, "message": message})

if resp.status_code == 200:
    print(f"[System] Webhook sent successfully. Job {JOB_ID} is now {status_to_report}.")
else:
    print(f"[Error] Webhook failed: {resp.status_code}")